# FarmTech Solutions — Fase 6: Visão Computacional
## Ir Além 2 — Transfer Learning × CNN do zero, com e sem segmentação

**Grupo:** Grupo 3
**Integrantes:** Gerson Ferreira da Graça (RM 569624), Lucas Braga (RM 568712), Carlos Leonardo Mazieri (RM 572809), Ryann Pinto (RM 571003)
**Objetos:** Tomate (classe 0) e Pimentão (classe 1)

> **Situação deste notebook:** estrutura inicial. As seções 1 a 3 (ambiente, dataset e segmentação) já têm código; os treinos (seções 4 a 6), a tabela final (7) e as conclusões (8) serão preenchidos na execução com GPU.

---

### 1. Objetivo e contexto

A Entrega 2 treinou uma **CNN do zero** para classificar a imagem inteira (tomate ou pimentão). Este notebook testa duas hipóteses sobre a mesma base de 80 imagens e a **mesma divisão** das Entregas 1 e 2 (32 treino / 4 validação / 4 teste por classe):

1. **Uma rede pré-treinada classifica melhor que uma treinada do zero?** — *Transfer Learning* com a **MobileNetV2** pré-treinada na ImageNet: as camadas convolucionais ficam congeladas e só o classificador final é treinado com as nossas fotos.
2. **Separar o objeto do fundo antes de classificar ajuda?** — **segmentação automática com GrabCut** (OpenCV): uma máscara do fruto é criada sem nenhuma rotulagem manual nova, o fundo é removido e a imagem é recortada no objeto antes de entrar na rede.

As duas hipóteses se cruzam em 4 combinações, todas avaliadas nas mesmas 8 imagens de teste:

| | **Sem segmentação** | **Com segmentação (GrabCut)** |
|:--|:--:|:--:|
| **CNN do zero** | resultado da Entrega 2 (não é treinada de novo) | seção 4 |
| **Transfer Learning (MobileNetV2)** | seção 5 | seção 6 |

**Por que MobileNetV2:** é leve (cerca de 2,2 milhões de parâmetros na parte convolucional), tem pesos da ImageNet prontos no Keras e aceita entrada de 128×128 — o mesmo tamanho usado pela CNN da Entrega 2, o que mantém a comparação justa.

**Por que GrabCut:** separa objeto e fundo a partir de um retângulo inicial, sem precisar de máscaras desenhadas à mão — o dataset tem caixas (YOLO), mas não máscaras.

## 1.1 Setup do ambiente

Use um ambiente com **GPU T4** (Ambiente de execução > Alterar tipo de ambiente de execução). Este notebook usa só TensorFlow e OpenCV, que já vêm instalados no Colab.

In [ ]:
# Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Caminho base do projeto no Drive (o mesmo das Entregas 1 e 2).
# Quem NÃO é o dono da pasta precisa, uma única vez, criar o atalho:
# "Compartilhados comigo" → botão direito em FarmTech_Fase6 → "Organizar" → "Adicionar atalho" → "Meu Drive".
BASE_PATH = '/content/drive/MyDrive/FarmTech_Fase6'

# Identificação de quem está rodando o notebook.
# Cada integrante troca para o próprio primeiro nome (minúsculo, sem acento) ANTES de rodar.
# Tudo o que este notebook grava vai para Resultados/ir_alem_2_<EXECUTOR>/.
EXECUTOR = 'gerson'
PASTA_IA2 = f'{BASE_PATH}/Resultados/ir_alem_2_{EXECUTOR}'

In [ ]:
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps

gpus = tf.config.list_physical_devices('GPU')
print('GPU disponível:', bool(gpus), gpus)
if not gpus:
    print('ATENÇÃO: sem GPU — troque para GPU T4 em Ambiente de execução > Alterar tipo de ambiente de execução.')
print('tensorflow', tf.__version__, '| opencv', cv2.__version__)

os.makedirs(PASTA_IA2, exist_ok=True)
print('Saídas deste notebook em:', PASTA_IA2)

## 2. Carregamento do dataset

As fotos são lidas direto das pastas por classe do Drive (`Dataset_dividido/{Tomate,Pimentao}/{Treino,Validacao,Teste}`) — a **mesma divisão** usada pela CNN da Entrega 2. Nenhuma imagem é copiada ou alterada no Drive.

In [ ]:
# 2.0 Conferir a divisão do dataset (falha cedo se algo estiver errado)
ORIGEM = os.path.join(BASE_PATH, 'Dataset_dividido')
CLASSES = ['tomate', 'pimentao']                       # índice 0 e 1 — mesma ordem das Entregas 1 e 2
PASTA_CLASSE = {'tomate': 'Tomate', 'pimentao': 'Pimentao'}
SPLITS = {'treino': 'Treino', 'validacao': 'Validacao', 'teste': 'Teste'}
ESPERADO = {'treino': 32, 'validacao': 4, 'teste': 4}  # por classe
EXTENSOES = ('.jpg', '.jpeg', '.png')


def nome_normalizado(arquivo):
    # 'Tomate 16.jpg' → 'tomate_16' (mesma regra das Entregas 1 e 2)
    return '_'.join(os.path.splitext(arquivo)[0].lower().split())


# Lista de (caminho, índice da classe, nome normalizado) por split
amostras = {split: [] for split in SPLITS}
problemas = []
for split, pasta_split in SPLITS.items():
    for indice, classe in enumerate(CLASSES):
        pasta = os.path.join(ORIGEM, PASTA_CLASSE[classe], pasta_split)
        arquivos = sorted(f for f in os.listdir(pasta) if f.lower().endswith(EXTENSOES))
        if len(arquivos) != ESPERADO[split]:
            problemas.append(f'{PASTA_CLASSE[classe]}/{pasta_split}: {len(arquivos)} imagens (esperado {ESPERADO[split]})')
        amostras[split] += [(os.path.join(pasta, f), indice, nome_normalizado(f)) for f in arquivos]
    print(f'{split:<9}: {len(amostras[split]):>2} imagens')

if problemas:
    raise RuntimeError('Problemas no dataset:\n- ' + '\n- '.join(problemas))

TESTE = amostras['teste']   # ordem fixa: 4 tomates, depois 4 pimentões — a mesma da Entrega 2
print('\n✅ Dataset conferido. Imagens de teste:', ', '.join(nome for _, _, nome in TESTE))

In [ ]:
# 2.1 Carregar as fotos originais em memória (a segmentação trabalha sobre a foto inteira)
TAMANHO = 128   # entrada das redes — o mesmo tamanho da CNN da Entrega 2


def carregar_original(caminho):
    """Foto em RGB (uint8), já na orientação correta (exif_transpose aplica a rotação gravada pelo celular)."""
    return np.asarray(ImageOps.exif_transpose(Image.open(caminho)).convert('RGB'))


def redimensionar(imagem):
    """Imagem RGB (qualquer tamanho) → array float32 TAMANHO×TAMANHO, pixels de 0 a 255."""
    return np.asarray(Image.fromarray(imagem).resize((TAMANHO, TAMANHO)), dtype=np.float32)


originais = {split: [carregar_original(c) for c, _, _ in lista] for split, lista in amostras.items()}
rotulos = {split: np.array([i for _, i, _ in lista], dtype=np.float32) for split, lista in amostras.items()}

# Versão SEM segmentação: a foto inteira redimensionada (exatamente o que a CNN da Entrega 2 recebeu)
X_sem = {split: np.stack([redimensionar(img) for img in imgs]) for split, imgs in originais.items()}
for split in SPLITS:
    print(f'{split:<9}: X {X_sem[split].shape} | pimentões: {int(rotulos[split].sum())}')

## 3. Segmentação automática com GrabCut

**Como funciona:** o GrabCut recebe um retângulo que contém o objeto e separa, pixel a pixel, o que é objeto e o que é fundo, modelando as cores de cada um. Aqui o retângulo é sempre o **centro da foto** (margem de 8% em cada borda), porque as fotos do dataset foram tiradas de perto, com o fruto no meio — assim nenhuma rotulagem manual é necessária.

**Etapas da função `segmentar_grabcut`:**
1. reduz a foto (lado maior = 320 px) para o GrabCut rodar rápido;
2. roda o GrabCut a partir do retângulo central (5 iterações, semente fixa — o resultado se repete a cada execução);
3. limpa a máscara: fecha pequenos buracos, fica só com a maior região conectada (o fruto) e preenche buracos internos (reflexos de luz);
4. volta a máscara ao tamanho original, pinta o fundo de preto e recorta a imagem no retângulo que envolve o objeto;
5. se a máscara cobrir menos de 5% da foto, considera que a segmentação falhou e devolve a foto original.

**Limitação esperada:** quando o fruto não está no centro, ou quando o fundo tem regiões de cor muito diferente (piso de tijolo, parede clara), o GrabCut pode pegar parte do fundo — ou o fundo no lugar do fruto. Por isso o teste visual abaixo inclui um caso em que isso acontece.

In [ ]:
# 3.0 Função de segmentação
def segmentar_grabcut(imagem, margem=0.08, iteracoes=5, lado_trabalho=320, area_minima=0.05):
    """Separa o objeto do fundo com GrabCut, sem rótulo manual.

    O GrabCut começa de um retângulo central (margem de 8% em cada borda) marcado como "provável objeto" —
    as fotos do dataset foram tiradas de perto, com o fruto no centro. Para ficar rápido, roda numa cópia
    reduzida (lado maior = 320 px) e a máscara volta ao tamanho original.

    Devolve (recortada, mascara, ok):
      recortada — imagem só com o objeto (fundo preto), cortada no retângulo que envolve a máscara
      mascara   — máscara binária (uint8, 0/1) no tamanho da imagem original
      ok        — False se a máscara ficou pequena demais (< 5% da foto); nesse caso a recortada é a original
    """
    altura, largura = imagem.shape[:2]
    escala = lado_trabalho / max(altura, largura)
    pequena = cv2.resize(imagem, (round(largura * escala), round(altura * escala)), interpolation=cv2.INTER_AREA)
    h, w = pequena.shape[:2]
    retangulo = (int(w * margem), int(h * margem), int(w * (1 - 2 * margem)), int(h * (1 - 2 * margem)))

    cv2.setRNGSeed(0)   # o GrabCut usa sorteios internos; semente fixa = mesmo resultado a cada execução
    mascara_gc = np.zeros((h, w), np.uint8)
    fundo, objeto = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
    cv2.grabCut(cv2.cvtColor(pequena, cv2.COLOR_RGB2BGR), mascara_gc, retangulo, fundo, objeto,
                iteracoes, cv2.GC_INIT_WITH_RECT)
    mascara = np.isin(mascara_gc, (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8)

    # Limpeza: fecha pequenos buracos e fica só com a maior região conectada (o fruto)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    n, rotulos, estat, _ = cv2.connectedComponentsWithStats(mascara)
    if n > 1:
        maior = 1 + int(np.argmax(estat[1:, cv2.CC_STAT_AREA]))
        mascara = (rotulos == maior).astype(np.uint8)

    # Preenche buracos internos (ex.: reflexos de luz no fruto, que o GrabCut às vezes marca como fundo)
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mascara = cv2.drawContours(np.zeros_like(mascara), contornos, -1, 1, thickness=cv2.FILLED)

    mascara = cv2.resize(mascara, (largura, altura), interpolation=cv2.INTER_NEAREST)
    if mascara.mean() < area_minima:
        return imagem, mascara, False

    ys, xs = np.where(mascara)
    so_objeto = imagem * mascara[:, :, None]
    return so_objeto[ys.min():ys.max() + 1, xs.min():xs.max() + 1], mascara, True


In [ ]:
# 3.1 Teste visual: original | máscara | recortada — 2 casos típicos e 1 caso difícil (fruto fora do centro)
EXEMPLOS = ['tomate_17', 'pimentao_36', 'tomate_29']
por_nome = {nome: (split, k) for split, lista in amostras.items() for k, (_, _, nome) in enumerate(lista)}

fig, eixos = plt.subplots(len(EXEMPLOS), 3, figsize=(10, 3.4 * len(EXEMPLOS)))
for linha, nome in enumerate(EXEMPLOS):
    split, k = por_nome[nome]
    recortada, mascara, ok = segmentar_grabcut(originais[split][k])
    for coluna, (imagem, titulo) in enumerate([(originais[split][k], f'{nome} — original'),
                                               (mascara, f'máscara ({mascara.mean():.0%} da foto)'),
                                               (recortada, 'recortada' + ('' if ok else ' (falhou: original)'))]):
        eixos[linha, coluna].imshow(imagem, cmap='gray' if imagem.ndim == 2 else None)
        eixos[linha, coluna].set_title(titulo, fontsize=10)
        eixos[linha, coluna].axis('off')
plt.tight_layout()
plt.savefig(f'{PASTA_IA2}/grabcut_exemplos.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Versão COM segmentação de todas as imagens (calculada uma vez e reaproveitada nas seções 4 e 6)
inicio = time.perf_counter()
segmentacao = {split: [segmentar_grabcut(img) for img in imgs] for split, imgs in originais.items()}
TEMPO_GRABCUT_TOTAL_S = time.perf_counter() - inicio

X_com = {split: np.stack([redimensionar(recortada) for recortada, _, _ in res]) for split, res in segmentacao.items()}
falhas = {split: [amostras[split][k][2] for k, (_, _, ok) in enumerate(res) if not ok] for split, res in segmentacao.items()}
n_imagens = sum(len(v) for v in originais.values())
print(f'GrabCut em {n_imagens} imagens: {TEMPO_GRABCUT_TOTAL_S:.1f} s (≈{TEMPO_GRABCUT_TOTAL_S / n_imagens * 1000:.0f} ms por imagem)')
print('Segmentações devolvidas como "falha" (máscara < 5%):', {s: f for s, f in falhas.items() if f} or 'nenhuma')

## 4. CNN do zero **com** segmentação

**Plano:** a mesma arquitetura e o mesmo treino da CNN da Entrega 2 — só muda a entrada, que passa a ser `X_com` (imagens recortadas pelo GrabCut):

- entrada 128×128×3; aumento de dados (flip, rotação 0,15, zoom 0,15); `Rescaling(1/255)`;
- 3 blocos `Conv2D` (16, 32, 64 filtros) + `MaxPooling2D`; `GlobalAveragePooling2D`; `Dropout(0.3)`; `Dense(1, sigmoid)`;
- Adam (1e-3), `binary_crossentropy`, até 100 épocas, `batch_size=8`, semente 0;
- parada antecipada por `val_accuracy` (paciência 30, restaurando os melhores pesos) — o mesmo critério corrigido da Entrega 2.

A combinação **CNN do zero sem segmentação** não é treinada de novo: usa o resultado registrado na Entrega 2 (7 de 8 no teste).

*(Código do treino: a implementar na execução com GPU.)*

## 5. Transfer Learning com MobileNetV2 — **sem** segmentação

**Plano:**

- base: `tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')`, com `base.trainable = False` — as camadas convolucionais ficam **congeladas** e mantêm o que aprenderam na ImageNet;
- pré-processamento: `tf.keras.applications.mobilenet_v2.preprocess_input` (pixels de 0–255 para −1 a 1, o formato em que a rede foi treinada) — no lugar do `Rescaling(1/255)` da CNN;
- o mesmo aumento de dados da CNN, antes da base;
- classificador treinado: `GlobalAveragePooling2D` → `Dropout(0.3)` → `Dense(1, sigmoid)`;
- treino com os mesmos critérios da seção 4 (Adam 1e-3, até 100 épocas, `batch_size=8`, semente 0, parada por `val_accuracy`), sobre `X_sem`.

*(Código do treino: a implementar na execução com GPU.)*

## 6. Transfer Learning com MobileNetV2 — **com** segmentação

**Plano:** exatamente o modelo e o treino da seção 5, trocando a entrada por `X_com` (imagens recortadas pelo GrabCut). Assim, a única diferença entre as seções 5 e 6 é a segmentação.

*(Código do treino: a implementar na execução com GPU.)*

## 7. Tabela comparativa final

As 4 combinações, medidas nas **mesmas 8 imagens de teste**:

| Abordagem | Segmentação | Acurácia no teste | Tempo de treino | Tempo de inferência (por imagem) |
|:--|:--:|:--:|:--:|:--:|
| CNN do zero | não | da Entrega 2 | da Entrega 2 | da Entrega 2 |
| CNN do zero | GrabCut | seção 4 | seção 4 | seção 4 (inclui o GrabCut) |
| MobileNetV2 (Transfer Learning) | não | seção 5 | seção 5 | seção 5 |
| MobileNetV2 (Transfer Learning) | GrabCut | seção 6 | seção 6 | seção 6 (inclui o GrabCut) |

**Como cada coluna será medida:**

- **acurácia:** acertos nas 8 imagens de teste, com limiar de 0,5 sobre a probabilidade de "pimentão" (o mesmo da Entrega 2);
- **tempo de treino:** só o `fit` (que inclui a validação a cada época), em GPU T4;
- **tempo de inferência:** mediana por imagem, uma imagem por vez — nas versões com segmentação, o tempo do GrabCut entra na conta, porque ele faz parte do caminho até a resposta;
- a linha da Entrega 2 vem de outra sessão do Colab: os números de tempo dela servem de referência, não de comparação exata.

*(Tabela: a preencher na execução com GPU.)*

## 8. Conclusões

A ser escrita com os números reais da execução, respondendo às duas hipóteses:

1. **A rede pré-treinada (MobileNetV2) classificou melhor que a CNN do zero?** — comparando as linhas de mesma segmentação.
2. **A pré-segmentação com GrabCut ajudou?** — comparando as colunas de mesma rede, e olhando imagem a imagem os casos em que o GrabCut pegou parte do fundo (seção 3).

Com só 8 imagens de teste, cada acerto vale 12,5 pontos percentuais — diferenças pequenas entre combinações não são conclusivas.